In [1]:
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report


In [2]:
df_train = pd.read_csv(r"D:\SER_Cross\data\processed\train.csv")
df_val   = pd.read_csv(r"D:\SER_Cross\data\processed\val.csv")
df_test  = pd.read_csv(r"D:\SER_Cross\data\processed\test.csv")

# 🔥 COMBINE EVERYTHING
df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)

print("Total samples:", len(df_all))
print(df_all["emotion"].value_counts())


Total samples: 7227
emotion
happy      1463
sad        1463
angry      1463
fear       1463
neutral    1375
Name: count, dtype: int64


In [3]:
SAMPLE_RATE = 22050
DURATION = 4
N_MELS = 128
MAX_LEN = 173

def spec_augment(mel):
    mel = mel.copy()
    t = mel.shape[1]
    t0 = np.random.randint(0, t - 20)
    mel[:, t0:t0+20] = 0
    return mel

def extract_logmel(path, augment=False):
    y, sr = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION)

    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=N_MELS, n_fft=1024, hop_length=512
    )

    logmel = librosa.power_to_db(mel)
    logmel = (logmel - logmel.mean()) / (logmel.std() + 1e-6)

    if augment:
        logmel = spec_augment(logmel)

    if logmel.shape[1] < MAX_LEN:
        logmel = np.pad(logmel, ((0,0),(0,MAX_LEN-logmel.shape[1])))
    else:
        logmel = logmel[:, :MAX_LEN]

    return logmel.T   # (time, mel)


In [4]:
X = np.array([extract_logmel(p, augment=False) for p in df_all["path"]])


c:\Users\Asus\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\Users\Asus\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
c:\Users\Asus\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


In [5]:
le = LabelEncoder()
y = le.fit_transform(df_all["emotion"])


In [6]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


Train: (5058, 173, 128)
Val: (1084, 173, 128)
Test: (1085, 173, 128)


In [7]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)


Class Weights: {0: 0.987890625, 1: 0.987890625, 2: 0.987890625, 3: 1.0515592515592516, 4: 0.987890625}


In [8]:
model = tf.keras.Sequential([
    layers.Conv1D(64, 5, activation="relu", input_shape=(173,128)),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),

    layers.Conv1D(128, 5, activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),

    layers.Bidirectional(layers.LSTM(128)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


c:\Users\Asus\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 169, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 169, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 84, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 80, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 80, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 40, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 379,589 (1.45 MB)

 Trainable params: 379,205 (1.45 MB)

 Non-trainable params: 384 (1.50 KB)

In [9]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    class_weight=class_weights
)


Epoch 1/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - accuracy: 0.4407 - loss: 1.3191 - val_accuracy: 0.4631 - val_loss: 1.2586
Epoch 2/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.5105 - loss: 1.1781 - val_accuracy: 0.4686 - val_loss: 1.2684
Epoch 3/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.5534 - loss: 1.0957 - val_accuracy: 0.4825 - val_loss: 1.2564
Epoch 4/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.5682 - loss: 1.0622 - val_accuracy: 0.4649 - val_loss: 1.2010
Epoch 5/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.6038 - loss: 0.9823 - val_accuracy: 0.5673 - val_loss: 1.0652
Epoch 6/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.6305 - loss: 0.9136 - val_accuracy: 0.5710 - val_loss: 1.0600
Epoch 7/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.6493 - loss: 0.8635 - val_accuracy: 0.5609 - val_loss: 1.1010
Epoch 8/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.6900 - loss: 0.7917 - val_accu

In [10]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Final Combined-Dataset Test Accuracy:", test_acc)


34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6516 - loss: 1.9503
Final Combined-Dataset Test Accuracy: 0.6516128778457642


In [11]:
y_pred = np.argmax(model.predict(X_test), axis=1)

print(classification_report(
    y_test, y_pred, target_names=le.classes_
))


34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
              precision    recall  f1-score   support

       angry       0.77      0.84      0.80       220
        fear       0.60      0.50      0.54       219
       happy       0.60      0.50      0.54       220
     neutral       0.64      0.75      0.69       206
         sad       0.62      0.68      0.65       220

    accuracy                           0.65      1085
   macro avg       0.65      0.65      0.65      1085
weighted avg       0.65      0.65      0.65      1085

